# Notebook 00 — Ingestão e Tratamento de Dados
## Subgrupo B3 Comercial — CELESC / ANEEL / CCEE

**Disciplina:** IAA68303 — Técnicas de Inteligência Artificial Aplicadas a Sistemas de Energia
**Programa:** Mestrado Profissional em Sistemas de Energia Elétrica (MPSEE) — IFSC
**Autor:** Dilsonei José Rigotti
**Professor:** Sérgio Luciano Ávila
**Relatório:** `report/Relatorio_Final_IEEE_TecnicasIA_Dilsonei.docx`

**Etapas do fluxo (Fig. 1 do relatório):** Ingestão → Visualização → **Tratamento (este notebook)** → Engenharia de dados → Modelagem → Avaliação

Ponto de entrada do pipeline. Este notebook:

1. Carrega os três conjuntos de dados brutos (consumo CELESC, tarifas ANEEL, PLD CCEE);
2. Aplica a filtragem e o saneamento descritos nas Seções III.A e III.B do relatório;
3. Exporta os arquivos tratados para `data/processed/`, lidos pelos notebooks 02 a 04. Nenhum notebook de modelagem lê os dados brutos diretamente.

### Organização dos dados

```text
data/
├── CELESC/      consumo por UC (bruto)            celesc_consumo_uc_1994_2026.csv.gz
├── ANEEL/       tarifas homologadas (bruto)       tarifas-homologadas-distribuidoras-energia-eletrica.csv.gz
├── CCEE/        PLD médio semanal (bruto)         pld_media_semanal_*.csv
└── processed/   saídas deste notebook             lidas pelos notebooks 02 a 04
```

Os brutos grandes são versionados comprimidos (`.csv.gz`) e lidos diretamente pelo `pandas`. Se houver apenas o `.csv` original na pasta, o notebook também o aceita.

## 1. Fontes de Dados Brutos

| Arquivo | Pasta | Fonte | Link | Data do download |
|---|---|---|---|---|
| `celesc_consumo_uc_1994_2026.csv.gz` | `data/CELESC/` | CELESC — Dados de Consumo (Mercado de Energia) | https://www.celesc.com.br/home/mercado-de-energia/dados-de-consumo | 02/08/2026 |
| `tarifas-homologadas-distribuidoras-energia-eletrica.csv.gz` | `data/ANEEL/` | ANEEL — Portal de Dados Abertos: *Tarifas de aplicação das distribuidoras de energia elétrica* | https://dadosabertos.aneel.gov.br/dataset/tarifas-distribuidoras-energia-eletrica | 07/09/2026 |
| `pld_media_semanal_*.csv` (2001-2026) | `data/CCEE/` | CCEE — Portal de Dados Abertos: *PLD Médio Semanal* | https://dadosabertos.ccee.org.br/dataset/pld_media_semanal | 12/09/2026 |

> As datas correspondem à data de gravação dos arquivos originais baixados dos portais.

> **Nota de reprodutibilidade:** portais públicos são atualizados continuamente (novas homologações tarifárias, novos PLDs semanais). Por isso os brutos usados nos resultados estão versionados no próprio repositório, e o arquivo `data/SHA256SUMS.txt` registra o hash (SHA-256) do conteúdo original de cada um — se alguém substituir um bruto por uma versão nova baixada do portal, o hash deixa de bater.

In [1]:
# --- Configuração de caminhos ---
# A raiz do projeto é localizada automaticamente (pasta que contém .git), de modo que o
# notebook funciona independentemente da pasta de trabalho em que o Jupyter foi aberto.
# Convenção: data/<FONTE>/ (brutos, .csv.gz) -> este notebook -> data/processed/ (tratados)

from pathlib import Path
import logging

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="[%(asctime)s] %(levelname)s - %(message)s", datefmt="%H:%M:%S")
logger = logging.getLogger("notebook0")


def localizar_raiz_projeto(marcador=".git"):
    for pasta in [Path.cwd(), *Path.cwd().parents]:
        if (pasta / marcador).exists():
            return pasta
    raise FileNotFoundError(f"Raiz do projeto (pasta com {marcador}) não encontrada a partir de {Path.cwd()}")


def localizar_bruto(pasta, nome_csv):
    """Retorna o .csv.gz versionado; aceita o .csv original como alternativa."""
    for candidato in (pasta / f"{nome_csv}.gz", pasta / nome_csv):
        if candidato.exists():
            return candidato
    raise FileNotFoundError(
        f"Bruto não encontrado: {pasta / (nome_csv + '.gz')}\n"
        "Rode `python scripts/organizar_dados.py` ou veja a seção 1 (Fontes de Dados Brutos)."
    )


RAIZ_PROJETO = localizar_raiz_projeto()
RAIZ_DADOS = RAIZ_PROJETO / "data"
DIR_CELESC = RAIZ_DADOS / "CELESC"
DIR_ANEEL = RAIZ_DADOS / "ANEEL"
DIR_CCEE = RAIZ_DADOS / "CCEE"
DIR_TRATADOS = RAIZ_DADOS / "processed"
DIR_TRATADOS.mkdir(parents=True, exist_ok=True)

CAMINHO_CELESC = localizar_bruto(DIR_CELESC, "celesc_consumo_uc_1994_2026.csv")
CAMINHO_TARIFAS = localizar_bruto(DIR_ANEEL, "tarifas-homologadas-distribuidoras-energia-eletrica.csv")
CAMINHOS_PLD = sorted(DIR_CCEE.glob("pld_media_semanal_*.csv*"))
if not CAMINHOS_PLD:
    raise FileNotFoundError(f"Nenhum pld_media_semanal_*.csv em {DIR_CCEE}")

logger.info("Raiz do projeto: %s (%s)", RAIZ_PROJETO.name, "encontrada pela pasta .git")
logger.info("CELESC: %s", CAMINHO_CELESC.relative_to(RAIZ_PROJETO))
logger.info("ANEEL: %s", CAMINHO_TARIFAS.relative_to(RAIZ_PROJETO))
logger.info("CCEE: %d arquivos de PLD", len(CAMINHOS_PLD))

[15:35:13] INFO - Raiz do projeto: Mercado_Energia_B3_Celesc (encontrada pela pasta .git)


[15:35:13] INFO - CELESC: data/CELESC/celesc_consumo_uc_1994_2026.csv.gz


[15:35:13] INFO - ANEEL: data/ANEEL/tarifas-homologadas-distribuidoras-energia-eletrica.csv.gz


[15:35:13] INFO - CCEE: 7 arquivos de PLD


## 2. Consumo CELESC — Isolamento da Classe Comercial

Ponto de partida comum às duas trilhas (Cativo e Livre): isola os registros da classe Comercial na base bruta de consumo (2.986.092 registros, 387 meses, jan/1994-mar/2026).

O filtro usa apenas a coluna `classe` (atividade econômica). "B3" é subgrupo tarifário e não é um valor dessa coluna; a separação Cativo/Livre é feita nas seções 3 e 4 pela coluna `tipo`. O log lista os valores de `classe` capturados e a distribuição por `tipo`, para conferência (esperado: 1.002.717 registros — 112.230 Cativo e 890.487 Livre).

In [2]:
COLUNAS_OBRIGATORIAS = {"data", "classe", "tipo", "cod_muni", "numero_uc", "consumo_mwh"}


def carregar_base_bruta(caminho_csv):
    df = pd.read_csv(caminho_csv, sep=",", encoding="utf-8-sig", low_memory=False)
    df.columns = [c.strip().lower() for c in df.columns]

    faltando = COLUNAS_OBRIGATORIAS - set(df.columns)
    if faltando:
        raise KeyError(f"Colunas ausentes: {sorted(faltando)}. Encontradas: {list(df.columns)}")

    for col in ["numero_uc", "consumo_mwh"]:
        n_nan_antes = df[col].isna().sum()
        df[col] = pd.to_numeric(df[col], errors="coerce")
        n_novos = df[col].isna().sum() - n_nan_antes
        if n_novos:
            logger.warning("%s: %d valores não numéricos convertidos em NaN", col, n_novos)

    logger.info("Base bruta carregada: %d registros, %d colunas", *df.shape)
    return df


def isolar_classe_comercial(df):
    classe = df["classe"].fillna("").astype(str).str.strip()
    mascara = classe.str.contains("comercial", case=False, regex=False)
    df_comercial = df[mascara].copy()
    logger.info("Classe Comercial isolada: %d de %d registros", len(df_comercial), len(df))
    logger.info("Valores de 'classe' capturados: %s", sorted(classe[mascara].unique()))
    logger.info("Distribuição por tipo: %s", df_comercial["tipo"].value_counts().to_dict())
    return df_comercial


def remover_duplicatas(df, rotulo):
    n_antes = len(df)
    df_sem_dup = df.drop_duplicates().copy()
    logger.info("[%s] Duplicatas removidas: %d (%d -> %d)", rotulo, n_antes - len(df_sem_dup), n_antes, len(df_sem_dup))
    return df_sem_dup


def preparar_datas(df, formato=None):
    # `formato` (ex.: "%Y-%m-%d") evita inversão dia/mês se a coluna vier como texto ambíguo
    df = df.copy()
    df["data"] = pd.to_datetime(df["data"], format=formato, errors="coerce")
    n_nat = int(df["data"].isna().sum())
    if n_nat:
        logger.warning("%d datas inválidas (NaT) descartadas", n_nat)
        df = df.dropna(subset=["data"])
    df["ano"] = df["data"].dt.year
    df["mes"] = df["data"].dt.month
    return df


df_bruto = carregar_base_bruta(CAMINHO_CELESC)
df_comercial = isolar_classe_comercial(df_bruto)

[15:35:18] INFO - Base bruta carregada: 2986092 registros, 12 colunas


[15:35:19] INFO - Classe Comercial isolada: 1002717 de 2986092 registros


[15:35:19] INFO - Valores de 'classe' capturados: ['Comercial']


[15:35:19] INFO - Distribuição por tipo: {'Livre': 890487, 'Cativo': 112230}


## 3. Trilha B3_Cativo

O Subgrupo B3 (baixa tensão) é 100% cativo por definição regulatória até a abertura do Mercado Livre em nov/2027 (Lei nº 15.269/2025). Etapas: isolamento do mercado Cativo → remoção de duplicatas → saneamento (`numero_uc <= 0`, `consumo_mwh < 0`) → arredondamento do `numero_uc` fracionário (out/2025-mar/2026, ver limitação documentada) → recálculo físico do consumo médio por UC → tratamento de outliers (IQR + limites físicos 100-3.000 kWh/UC/mês) → agregação mensal (com a correção metodológica: `numero_uc` agregado a partir da base saneada, **antes** do filtro de outlier de consumo, para não introduzir o artefato sazonal de jan/fev documentado no relatório).


In [3]:
LIMITE_FISICO_INFERIOR = 100.0
LIMITE_FISICO_SUPERIOR = 3000.0
IQR_MULTIPLICADOR_CATIVO = 1.5


def processar_trilha_cativo(df_comercial):
    df_cativo = df_comercial[df_comercial["tipo"] == "Cativo"].copy()
    df_cativo = remover_duplicatas(df_cativo, "Cativo")
    df_cativo = preparar_datas(df_cativo)

    df_saneado = df_cativo[(df_cativo["numero_uc"] > 0) & (df_cativo["consumo_mwh"] >= 0)].copy()

    # Correção de exibição: numero_uc fracionário em out/2025-mar/2026 (mudança de
    # metodologia de apuração da CELESC nesse intervalo — ver relatório, limitação documentada)
    n_fracionario = int((df_saneado["numero_uc"] % 1 != 0).sum())
    if n_fracionario:
        logger.info("numero_uc fracionário arredondado em %d registros", n_fracionario)
        df_saneado["numero_uc"] = df_saneado["numero_uc"].round()

    df_saneado["consumo_kwh_uc_corrigido"] = (df_saneado["consumo_mwh"] * 1000) / df_saneado["numero_uc"]

    q1 = df_saneado["consumo_kwh_uc_corrigido"].quantile(0.25)
    q3 = df_saneado["consumo_kwh_uc_corrigido"].quantile(0.75)
    iqr = q3 - q1
    lim_inf = max(0.0, q1 - IQR_MULTIPLICADOR_CATIVO * iqr, LIMITE_FISICO_INFERIOR)
    lim_sup = min(q3 + IQR_MULTIPLICADOR_CATIVO * iqr, LIMITE_FISICO_SUPERIOR)

    df_tratado = df_saneado[
        (df_saneado["consumo_kwh_uc_corrigido"] >= lim_inf) & (df_saneado["consumo_kwh_uc_corrigido"] <= lim_sup)
    ].copy()
    logger.info("[Cativo] IQR+limites físicos: [%.2f, %.2f] kWh/UC/mês, %d outliers removidos",
                lim_inf, lim_sup, len(df_saneado) - len(df_tratado))

    # numero_uc a partir da base SANEADA (pré-IQR) -- ver correção metodológica no relatório
    numero_uc_mensal = df_saneado.groupby(["ano", "mes"])["numero_uc"].sum().reset_index()

    df_mensal = df_tratado.groupby(["ano", "mes"]).agg(
        consumo_mwh=("consumo_mwh", "sum"),
        consumo_kwh_uc_corrigido=("consumo_kwh_uc_corrigido", "mean"),
    ).reset_index()
    df_mensal = pd.merge(df_mensal, numero_uc_mensal, on=["ano", "mes"], how="outer")
    df_mensal = df_mensal.sort_values(["ano", "mes"]).reset_index(drop=True)
    logger.info("[Cativo] Série mensal: %d meses (esperado 376 de 387 -- jan-nov/1994 sem numero_uc válido)", len(df_mensal))
    return df_mensal


df_mensal_cativo = processar_trilha_cativo(df_comercial)
df_mensal_cativo.tail()


[15:35:19] INFO - [Cativo] Duplicatas removidas: 0 (112230 -> 112230)


[15:35:19] INFO - numero_uc fracionário arredondado em 1624 registros


[15:35:19] INFO - [Cativo] IQR+limites físicos: [100.00, 1484.53] kWh/UC/mês, 5409 outliers removidos


[15:35:19] INFO - [Cativo] Série mensal: 376 meses (esperado 376 de 387 -- jan-nov/1994 sem numero_uc válido)


,ano,mes,consumo_mwh,consumo_kwh_uc_corrigido,numero_uc
371,2025,11,221806.494385,504.532149,342962.0
372,2025,12,230805.837001,523.616331,342812.0
373,2026,1,270675.630533,590.392594,345179.0
374,2026,2,274307.306010,581.116488,342969.0
375,2026,3,298529.444757,640.022751,346466.0


## 4. Trilha Comercial_Livre_ACL (referência/comparação — não é B3)

Consumidores comerciais de alta tensão (Grupo A) já migrados ao Mercado Livre. Usados como série de referência para observar comportamento pós-migração, já que o B3 ainda não pode migrar. Etapa crítica: remoção de duplicatas (82,6% das linhas brutas deste subconjunto são duplicatas exatas) e correção do *broadcast* de `numero_uc` (cada linha é um cliente individual; `numero_uc` traz o total do município/mês repetido em cada linha — não pode ser somado linha a linha).


In [4]:
IQR_MULTIPLICADOR_LIVRE = 3.0


def processar_trilha_livre(df_comercial):
    df_livre = df_comercial[df_comercial["tipo"] == "Livre"].copy()
    df_livre = remover_duplicatas(df_livre, "Livre")
    df_livre = preparar_datas(df_livre)

    df_saneado = df_livre[df_livre["consumo_mwh"] >= 0].copy()

    # numero_uc: 1 valor por (município, mês) -- corrige o broadcast (ver Seção 3.3 do relatório)
    uc_por_municipio_mes = df_saneado.groupby(["cod_muni", "ano", "mes"])["numero_uc"].first().reset_index()
    numero_uc_mensal = uc_por_municipio_mes.groupby(["ano", "mes"])["numero_uc"].sum().reset_index()

    coluna = "consumo_mwh"
    q1, q3 = df_saneado[coluna].quantile([0.25, 0.75])
    iqr = q3 - q1
    lim_inf = max(0.0, q1 - IQR_MULTIPLICADOR_LIVRE * iqr)
    lim_sup = q3 + IQR_MULTIPLICADOR_LIVRE * iqr
    df_tratado = df_saneado[(df_saneado[coluna] >= lim_inf) & (df_saneado[coluna] <= lim_sup)].copy()

    consumo_mensal = df_tratado.groupby(["ano", "mes"])["consumo_mwh"].sum().reset_index()
    df_mensal = pd.merge(consumo_mensal, numero_uc_mensal, on=["ano", "mes"], how="outer")
    df_mensal = df_mensal.sort_values(["ano", "mes"]).reset_index(drop=True)
    logger.info("[Livre] Série mensal: %d meses", len(df_mensal))
    return df_mensal


df_mensal_livre = processar_trilha_livre(df_comercial)
df_mensal_livre.tail()


[15:35:20] INFO - [Livre] Duplicatas removidas: 735548 (890487 -> 154939)


[15:35:20] INFO - [Livre] Série mensal: 387 meses


,ano,mes,consumo_mwh,numero_uc
382,2025,11,132951.205082,2030.0
383,2025,12,153835.931701,2078.0
384,2026,1,159195.776348,2170.0
385,2026,2,155500.878454,2205.0
386,2026,3,169519.283365,2243.0


## 5. Tarifas B3 CELESC (ANEEL)

Filtra a base nacional de tarifas homologadas (327.493 registros) para CELESC, Subgrupo B3, modalidade Convencional, excluindo registros de compensação de geração distribuída (SCEE) e mantendo apenas a Tarifa de Aplicação (o valor efetivamente cobrado do consumidor). Expande cada período de vigência em uma série mensal de TUSD e TE.


In [5]:
def tratar_tarifas_b3_celesc(caminho_csv, data_inicio="2010-08-01", data_fim="2027-08-01"):
    df = pd.read_csv(caminho_csv, sep=";", encoding="utf-8", dtype=str)
    cel = df[df["SigAgente"] == "CELESC"].copy()
    b3 = cel[cel["DscSubGrupo"] == "B3"]
    conv = b3[
        (b3["DscModalidadeTarifaria"] == "Convencional") &
        (b3["NomPostoTarifario"] == "Não se aplica") &
        (b3["DscDetalhe"] == "Não se aplica") &
        (b3["DscBaseTarifaria"] == "Tarifa de Aplicação")
    ].copy()
    conv["DatInicioVigencia"] = pd.to_datetime(conv["DatInicioVigencia"])
    conv["DatFimVigencia"] = pd.to_datetime(conv["DatFimVigencia"])
    conv["VlrTUSD"] = conv["VlrTUSD"].str.replace(",", ".").astype(float)
    conv["VlrTE"] = conv["VlrTE"].str.replace(",", ".").astype(float)
    conv = conv.sort_values("DatInicioVigencia").reset_index(drop=True)

    meses = pd.date_range(data_inicio, data_fim, freq="MS")
    linhas = []
    for m in meses:
        linha = conv[(conv["DatInicioVigencia"] <= m) & (conv["DatFimVigencia"] >= m)]
        if len(linha) == 0:
            tusd, te = np.nan, np.nan
        else:
            tusd, te = linha.iloc[0]["VlrTUSD"], linha.iloc[0]["VlrTE"]
        linhas.append({"data": m, "tusd_b3_rs_mwh": tusd, "te_b3_rs_mwh": te})
    tarifa_mensal = pd.DataFrame(linhas)
    tarifa_mensal["tarifa_cativo_rs_mwh"] = tarifa_mensal["tusd_b3_rs_mwh"] + tarifa_mensal["te_b3_rs_mwh"]
    logger.info("Tarifa B3 CELESC mensal: %d meses (%s a %s)", len(tarifa_mensal), meses.min().date(), meses.max().date())
    return tarifa_mensal


tarifa_b3_mensal = tratar_tarifas_b3_celesc(CAMINHO_TARIFAS)
tarifa_b3_mensal.tail()


[15:35:21] INFO - Tarifa B3 CELESC mensal: 205 meses (2010-08-01 a 2027-08-01)


,data,tusd_b3_rs_mwh,te_b3_rs_mwh,tarifa_cativo_rs_mwh
200,2027-04-01,414.04,345.71,759.75
201,2027-05-01,414.04,345.71,759.75
202,2027-06-01,414.04,345.71,759.75
203,2027-07-01,414.04,345.71,759.75
204,2027-08-01,414.04,345.71,759.75


## 6. PLD Sul (CCEE)

Consolida os arquivos semanais de PLD (2001-2026), filtra o submercado Sul e agrega em médias mensais — usado como proxy do preço de energia no Ambiente de Contratação Livre (ACL) na análise financeira (notebook 04).


In [6]:
def tratar_pld_sul(caminhos_csv):
    dfs = [pd.read_csv(c, sep=";", encoding="utf-8", dtype=str) for c in caminhos_csv]
    pld = pd.concat(dfs, ignore_index=True)
    pld.columns = [c.strip() for c in pld.columns]
    pld["SEMANA"] = pd.to_datetime(pld["SEMANA"], format="%d/%m/%Y")
    pld["PLD_MEDIA_SEMANA"] = pld["PLD_MEDIA_SEMANA"].str.replace(",", ".").astype(float)
    pld_sul = pld[pld["SUBMERCADO"] == "SUL"].sort_values("SEMANA").drop_duplicates(subset="SEMANA")
    pld_mensal = pld_sul.set_index("SEMANA")["PLD_MEDIA_SEMANA"].resample("MS").mean().reset_index()
    pld_mensal.columns = ["data", "pld_sul_rs_mwh"]
    logger.info("PLD Sul mensal: %d meses (%s a %s)", len(pld_mensal), pld_mensal["data"].min().date(), pld_mensal["data"].max().date())
    return pld_mensal


pld_sul_mensal = tratar_pld_sul(CAMINHOS_PLD)
pld_sul_mensal.tail()


[15:35:21] INFO - PLD Sul mensal: 304 meses (2001-06-01 a 2026-09-01)


,data,pld_sul_rs_mwh
299,2026-05-01,239.3700
300,2026-06-01,184.3125
301,2026-07-01,136.5475
302,2026-08-01,135.6280
303,2026-09-01,130.2600


## 7. Exportação dos Arquivos Tratados

Arquivos gravados em `data/processed/` e lidos pelos notebooks 02 a 04. A série B3 Cativo é exportada em dois formatos: `b3_cativo_mensal.csv` (lido pelos notebooks) e `Subgrupo_B3_Comercial_Tratado_1994_2026_B3_Cativo.csv` (separador `;` e decimal `,`, para abrir no Excel).

In [7]:
df_mensal_cativo.to_csv(DIR_TRATADOS / "b3_cativo_mensal.csv", index=False)
df_mensal_livre.to_csv(DIR_TRATADOS / "comercial_livre_acl_mensal.csv", index=False)
tarifa_b3_mensal.to_csv(DIR_TRATADOS / "tarifa_b3_celesc_mensal.csv", index=False)
pld_sul_mensal.to_csv(DIR_TRATADOS / "pld_sul_mensal.csv", index=False)

# Cópia da série B3 Cativo com separador ';' e decimal ',' (abre direto no Excel pt-BR)
df_mensal_cativo.to_csv(
    DIR_TRATADOS / "Subgrupo_B3_Comercial_Tratado_1994_2026_B3_Cativo.csv",
    index=False, sep=";", decimal=",", encoding="utf-8-sig",
)
logger.info("Série B3 Cativo exportada também no formato Excel pt-BR (sep=';', decimal=',')")

for arq in DIR_TRATADOS.glob("*.csv"):
    logger.info("Exportado: %s (%.1f KB)", arq.name, arq.stat().st_size / 1024)


[15:35:21] INFO - Série B3 Cativo exportada também no formato Excel pt-BR (sep=';', decimal=',')


[15:35:21] INFO - Exportado: comparativo_custo_b3_cativo_vs_livre_2030.csv (8.1 KB)


[15:35:21] INFO - Exportado: simulacao_financeira_cativo_livre_2030.csv (7.3 KB)


[15:35:21] INFO - Exportado: tarifa_b3_celesc_mensal.csv (7.1 KB)


[15:35:21] INFO - Exportado: b3_cativo_mensal.csv (17.0 KB)


...

[15:35:21] INFO - Exportado: sc_grupo_b_mensal.csv (59.4 KB)


[15:35:21] INFO - Exportado: pld_sul_mensal.csv (6.1 KB)


[15:35:21] INFO - Exportado: Subgrupo_B3_Comercial_Tratado_1994_2026_B3_Cativo.csv (17.0 KB)


[15:35:21] INFO - Exportado: walkforward_mape_por_janela.csv (1.4 KB)


[15:35:21] INFO - Exportado: comercial_livre_acl_mensal.csv (7.8 KB)


## 8. Resumo das Inconsistências Tratadas

| Inconsistência | Ação aplicada |
|---|---|
| Divergência consumo x consumo médio pré-calculado | Descarte `numero_uc<=0`; recálculo físico via `(MWh×1000)/numero_uc` |
| Picos atípicos no consumo médio por UC | Filtro IQR + limites físicos (100-3.000 kWh/UC/mês), trilha Cativo |
| Mistura de mercados Cativo e Livre sob a mesma classe | Separação em duas trilhas paralelas |
| 82,6% de duplicatas exatas no subconjunto Livre+Comercial | `drop_duplicates()` antes de qualquer agregação |
| `numero_uc` broadcast por (município,mês) na trilha Livre | Agregação por 1 valor por grupo antes de somar |
| Lacuna total de `numero_uc` em jan-nov/1994 (todas as classes) | Período excluído da série tratada; documentado, não corrigido |
| `numero_uc` fracionário em out/2025-mar/2026 | Arredondamento para inteiro; limitação documentada |
| Artefato sazonal na série de UCs (quedas em jan/fev) | `numero_uc` agregado a partir da base saneada, antes do IQR |

Detalhamento de cada item: Seção III.A do relatório.

**Próximo notebook do pipeline B3:** `02_comparacao_modelos_b3.ipynb`.

Observação: O notebook 01 é uma análise exploratória independente do Grupo B e pode ser lido a qualquer momento.
